In [ ]:
# ============================================================
# Install FFmpeg
# ============================================================

!apt-get update -qq
!apt-get install -qq ffmpeg

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import subprocess

from pathlib import Path
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

print(os.listdir("/content/drive/MyDrive"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['Colab Notebooks', 'Getting started.pdf', 'Untitled presentation.gslides', 'Untitled spreadsheet (1).gsheet', 'Flash cards.gslides', 'IMG_20210325_145213_653.jpg', 'Patrika,Hindi.pdf', 'Patrika, hindi.pdf', 'Patrika Hindi.pdf', 'Patrika, hindi10.33 AM.pdf', 'COVID-19 certificate.pdf', 'Priyam Chaudhary.gdoc', 'IMG_6122.JPG', 'IMG_3920.JPG', 'IMG_5240.JPG', 'PROFORMA-TIET-DOAA-SCHMCM.pdf', 'AFFIDAVIT FOR DECLARING FAMILY(MCM).pdf', 'Income (1).pdf', 'Scholarship D.pdf', 'Scholar pdf.pdf', 'Hostel Fee-Receipt (1).pdf', 'Mess Fee-Receipt (1).pdf', 'b2bffae5-17bb-4cd3-80d5-4face3b3ded9.jpg', 'IMG_8047.jpeg', 'Mess Fee-Receipt.pdf', 'Hostel Fee-Receipt.pdf', 'Workout.pdf', 'Edp assign3.pdf', 'acer Laptop Bill (1).pdf', 'NIT KKR.pdf', '3rd sem fee.pdf', 'Income.pdf', 'lostreport442699_ssl.pdf', '_Registration_Form_BE_BTECH_120153 (1).pdf', 'OD225570703328609000.pd

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/SentinalMAE"))

['train', 'test']


In [ ]:
# ============================================================
# Configuration
# ============================================================

DATASET_ROOT = "/content/drive/MyDrive/SentinalMAE"

# Save locally in Colab
OUTPUT_ROOT = "/content/SentinelMAE_Processed"

CLASSES = [
    "Fighting",
    "Normal",
    "Shooting"
]

SPLITS = [
    "train",
    "test"
]

In [ ]:
from pathlib import Path

DATASET_ROOT = "/content/drive/MyDrive/SentinalMAE"   # apna path yahan likho

print("Train folders:")
print(list((Path(DATASET_ROOT) / "train").iterdir()))

print("\nTest folders:")
print(list((Path(DATASET_ROOT) / "test").iterdir()))

Train folders:
[PosixPath('/content/drive/MyDrive/SentinalMAE/train/Normal'), PosixPath('/content/drive/MyDrive/SentinalMAE/train/Shooting'), PosixPath('/content/drive/MyDrive/SentinalMAE/train/Fighting')]

Test folders:
[PosixPath('/content/drive/MyDrive/SentinalMAE/test/Normal'), PosixPath('/content/drive/MyDrive/SentinalMAE/test/Shooting'), PosixPath('/content/drive/MyDrive/SentinalMAE/test/Fighting')]


In [ ]:
from pathlib import Path

audio_files = list(Path("/content/SentinelMAE_Processed").rglob("*.wav"))

print("Total Audio Files:", len(audio_files))

print(audio_files[:5])

Total Audio Files: 3122
[PosixPath('/content/SentinelMAE_Processed/train/Normal/audio/normal_000409.wav'), PosixPath('/content/SentinelMAE_Processed/train/Normal/audio/normal_001242.wav'), PosixPath('/content/SentinelMAE_Processed/train/Normal/audio/normal_000456.wav'), PosixPath('/content/SentinelMAE_Processed/train/Normal/audio/normal_001391.wav'), PosixPath('/content/SentinelMAE_Processed/train/Normal/audio/normal_000778.wav')]


In [ ]:
# ============================================================
# Create Audio Folders
# ============================================================

for split in SPLITS:

    for cls in CLASSES:

        Path(
            OUTPUT_ROOT,
            split,
            cls,
            "audio"
        ).mkdir(
            parents=True,
            exist_ok=True
        )

print("Folders Created Successfully")

Folders Created Successfully


In [ ]:
# ============================================================
# Audio Extraction Function
# ============================================================

def extract_audio(video_path, output_path):

    command = [

        "ffmpeg",

        "-y",

        "-i",
        str(video_path),

        "-vn",

        "-ac",
        "1",

        "-ar",
        "16000",

        "-acodec",
        "pcm_s16le",

        str(output_path)

    ]

    subprocess.run(

        command,

        stdout=subprocess.DEVNULL,

        stderr=subprocess.DEVNULL

    )

In [ ]:
# ============================================================
# Process Complete Dataset
# ============================================================

processed = 0
skipped = 0
failed = 0

failed_files = []

for split in SPLITS:

    print("\n"+"="*60)
    print(split.upper())
    print("="*60)

    for cls in CLASSES:

        print(f"\nProcessing {cls}")

        input_folder = Path(DATASET_ROOT)/split/cls

        output_folder = Path(
            OUTPUT_ROOT,
            split,
            cls,
            "audio"
        )

        output_folder.mkdir(
            parents=True,
            exist_ok=True
        )

        videos=[]

        for ext in [
            "*.mp4",
            "*.avi",
            "*.mov",
            "*.mkv"
        ]:

            videos.extend(
                sorted(
                    input_folder.glob(ext)
                )
            )

        for idx, video in enumerate(
            tqdm(
                videos,
                desc=f"{split}/{cls}"
            )
        ):

            save_path = output_folder / f"{cls.lower()}_{idx+1:06d}.wav"

            if save_path.exists():

                skipped += 1

                continue

            try:

                extract_audio(
                    video,
                    save_path
                )

                processed += 1

            except Exception as e:

                failed += 1

                failed_files.append(
                    str(video)
                )

                print(video.name)

print("\n")
print("="*60)

print("Finished")

print("="*60)

print("Processed :",processed)

print("Skipped :",skipped)

print("Failed :",failed)

with open(
    "/content/failed_audio.txt",
    "w"
) as f:

    for file in failed_files:

        f.write(file+"\n")

print("\nFailed Log Saved")


TRAIN

Processing Fighting


train/Fighting: 100%|██████████| 377/377 [00:00<00:00, 11631.99it/s]


Processing Normal



train/Normal: 100%|██████████| 2049/2049 [00:09<00:00, 221.45it/s]



Processing Shooting


...


Processing Normal


test/Normal: 100%|██████████| 300/300 [00:00<00:00, 26215.49it/s]



Processing Shooting


test/Shooting: 100%|██████████| 62/62 [00:00<00:00, 11340.41it/s]



Finished
Processed : 5
Skipped : 3122
Failed : 0

Failed Log Saved


In [ ]:
from pathlib import Path
import random

files = list(
    Path("/content/SentinelMAE_Processed").rglob("*.wav")
)

print("Total Audio Files :",len(files))

sample = random.choice(files)

print(sample)

Total Audio Files : 3126
/content/SentinelMAE_Processed/train/Normal/audio/normal_001210.wav


In [ ]:
from pathlib import Path

print("Audio files:",
      len(list(Path("/content/SentinelMAE_Processed").rglob("*.wav"))))

Audio files: 3126


In [ ]:
!du -sh /content/SentinelMAE_Processed/train/Normal/audio
!du -sh /content/SentinelMAE_Processed/train/Fighting/audio
!du -sh /content/SentinelMAE_Processed/train/Shooting/audio
!du -sh /content/SentinelMAE_Processed/test/Normal/audio
!du -sh /content/SentinelMAE_Processed/test/Fighting/audio
!du -sh /content/SentinelMAE_Processed/test/Shooting/audio

14G	/content/SentinelMAE_Processed/train/Normal/audio
1.2G	/content/SentinelMAE_Processed/train/Fighting/audio
364M	/content/SentinelMAE_Processed/train/Shooting/audio
1.6G	/content/SentinelMAE_Processed/test/Normal/audio
313M	/content/SentinelMAE_Processed/test/Fighting/audio
98M	/content/SentinelMAE_Processed/test/Shooting/audio


In [ ]:
!zip -r /content/train_normal_audio.zip /content/SentinelMAE_Processed/train/Normal/audio

  adding: content/SentinelMAE_Processed/train/Normal/audio/ (stored 0%)
  adding: content/SentinelMAE_Processed/train/Normal/audio/normal_000409.wav (deflated 17%)
  adding: content/SentinelMAE_Processed/train/Normal/audio/normal_001242.wav (deflated 8%)
  adding: content/SentinelMAE_Processed/train/Normal/audio/normal_000456.wav (deflated 28%)
  adding: content/SentinelMAE_Processed/train/Normal/audio/normal_001391.wav (deflated 4%)
  adding: content/SentinelMAE_Processed/train/Normal/audio/normal_000778.wav (deflated 38%)
  adding: content/SentinelMAE_Processed/train/Normal/audio/normal_000488.wav (deflated 29%)
  adding: content/SentinelMAE_Processed/train/Normal/audio/normal_000536.wav (deflated 33%)
  adding: content/SentinelMAE_Processed/train/Normal/audio/normal_001416.wav (deflated 10%)
  adding: content/SentinelMAE_Processed/train/Normal/audio/normal_001938.wav (deflated 13%)
  adding: content/SentinelMAE_Processed/train/Normal/audio/normal_001298.wav (deflated 14%)
  adding: 

In [ ]:
from google.colab import files
files.download("/content/train_normal_audio.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -r /content/train_fighting_audio.zip /content/SentinelMAE_Processed/train/Fighting/audio

  adding: content/SentinelMAE_Processed/train/Fighting/audio/ (stored 0%)
  adding: content/SentinelMAE_Processed/train/Fighting/audio/fighting_000147.wav (deflated 19%)
  adding: content/SentinelMAE_Processed/train/Fighting/audio/fighting_000287.wav (deflated 8%)
  adding: content/SentinelMAE_Processed/train/Fighting/audio/fighting_000033.wav (deflated 22%)
  adding: content/SentinelMAE_Processed/train/Fighting/audio/fighting_000341.wav (deflated 9%)
  adding: content/SentinelMAE_Processed/train/Fighting/audio/fighting_000162.wav (deflated 13%)
  adding: content/SentinelMAE_Processed/train/Fighting/audio/fighting_000233.wav (deflated 9%)
  adding: content/SentinelMAE_Processed/train/Fighting/audio/fighting_000107.wav (deflated 13%)
  adding: content/SentinelMAE_Processed/train/Fighting/audio/fighting_000113.wav (deflated 27%)
  adding: content/SentinelMAE_Processed/train/Fighting/audio/fighting_000319.wav (deflated 10%)
  adding: content/SentinelMAE_Processed/train/Fighting/audio/figh

In [ ]:
from google.colab import files
files.download("/content/train_fighting_audio.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -r /content/train_shooting_audio.zip /content/SentinelMAE_Processed/train/Shooting/audio

  adding: content/SentinelMAE_Processed/train/Shooting/audio/ (stored 0%)
  adding: content/SentinelMAE_Processed/train/Shooting/audio/shooting_000062.wav (deflated 36%)
  adding: content/SentinelMAE_Processed/train/Shooting/audio/shooting_000223.wav (deflated 15%)
  adding: content/SentinelMAE_Processed/train/Shooting/audio/shooting_000091.wav (deflated 19%)
  adding: content/SentinelMAE_Processed/train/Shooting/audio/shooting_000036.wav (deflated 28%)
  adding: content/SentinelMAE_Processed/train/Shooting/audio/shooting_000033.wav (deflated 34%)
  adding: content/SentinelMAE_Processed/train/Shooting/audio/shooting_000071.wav (deflated 13%)
  adding: content/SentinelMAE_Processed/train/Shooting/audio/shooting_000151.wav (deflated 13%)
  adding: content/SentinelMAE_Processed/train/Shooting/audio/shooting_000191.wav (deflated 16%)
  adding: content/SentinelMAE_Processed/train/Shooting/audio/shooting_000139.wav (deflated 15%)
  adding: content/SentinelMAE_Processed/train/Shooting/audio/s

In [ ]:
from google.colab import files
files.download("/content/train_shooting_audio.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -r /content/test_normal_audio.zip /content/SentinelMAE_Processed/test/Normal/audio

  adding: content/SentinelMAE_Processed/test/Normal/audio/ (stored 0%)
  adding: content/SentinelMAE_Processed/test/Normal/audio/normal_000173.wav (deflated 13%)
  adding: content/SentinelMAE_Processed/test/Normal/audio/normal_000103.wav (deflated 27%)
  adding: content/SentinelMAE_Processed/test/Normal/audio/normal_000149.wav (deflated 32%)
  adding: content/SentinelMAE_Processed/test/Normal/audio/normal_000078.wav (deflated 24%)
  adding: content/SentinelMAE_Processed/test/Normal/audio/normal_000251.wav (deflated 12%)
  adding: content/SentinelMAE_Processed/test/Normal/audio/normal_000190.wav (deflated 5%)
  adding: content/SentinelMAE_Processed/test/Normal/audio/normal_000052.wav (deflated 26%)
  adding: content/SentinelMAE_Processed/test/Normal/audio/normal_000071.wav (deflated 14%)
  adding: content/SentinelMAE_Processed/test/Normal/audio/normal_000003.wav (deflated 41%)
  adding: content/SentinelMAE_Processed/test/Normal/audio/normal_000091.wav (deflated 36%)
  adding: content/Se

In [ ]:
from google.colab import files
files.download("/content/test_normal_audio.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -r /content/test_fighting_audio.zip /content/SentinelMAE_Processed/test/Fighting/audio

  adding: content/SentinelMAE_Processed/test/Fighting/audio/ (stored 0%)
  adding: content/SentinelMAE_Processed/test/Fighting/audio/fighting_000033.wav (deflated 18%)
  adding: content/SentinelMAE_Processed/test/Fighting/audio/fighting_000107.wav (deflated 31%)
  adding: content/SentinelMAE_Processed/test/Fighting/audio/fighting_000012.wav (deflated 19%)
  adding: content/SentinelMAE_Processed/test/Fighting/audio/fighting_000076.wav (deflated 12%)
  adding: content/SentinelMAE_Processed/test/Fighting/audio/fighting_000084.wav (deflated 15%)
  adding: content/SentinelMAE_Processed/test/Fighting/audio/fighting_000042.wav (deflated 27%)
  adding: content/SentinelMAE_Processed/test/Fighting/audio/fighting_000061.wav (deflated 18%)
  adding: content/SentinelMAE_Processed/test/Fighting/audio/fighting_000072.wav (deflated 11%)
  adding: content/SentinelMAE_Processed/test/Fighting/audio/fighting_000073.wav (deflated 12%)
  adding: content/SentinelMAE_Processed/test/Fighting/audio/fighting_000

In [ ]:
from google.colab import files
files.download("/content/test_fighting_audio.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -r /content/test_shooting_audio.zip /content/SentinelMAE_Processed/test/Shooting/audio

  adding: content/SentinelMAE_Processed/test/Shooting/audio/ (stored 0%)
  adding: content/SentinelMAE_Processed/test/Shooting/audio/shooting_000062.wav (deflated 24%)
  adding: content/SentinelMAE_Processed/test/Shooting/audio/shooting_000036.wav (deflated 17%)
  adding: content/SentinelMAE_Processed/test/Shooting/audio/shooting_000033.wav (deflated 10%)
  adding: content/SentinelMAE_Processed/test/Shooting/audio/shooting_000025.wav (deflated 29%)
  adding: content/SentinelMAE_Processed/test/Shooting/audio/shooting_000012.wav (deflated 29%)
  adding: content/SentinelMAE_Processed/test/Shooting/audio/shooting_000031.wav (deflated 17%)
  adding: content/SentinelMAE_Processed/test/Shooting/audio/shooting_000034.wav (deflated 19%)
  adding: content/SentinelMAE_Processed/test/Shooting/audio/shooting_000061.wav (deflated 22%)
  adding: content/SentinelMAE_Processed/test/Shooting/audio/shooting_000037.wav (deflated 15%)
  adding: content/SentinelMAE_Processed/test/Shooting/audio/shooting_000

In [ ]:
from google.colab import files
files.download("/content/test_shooting_audio.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>